# Sprint 2 — retrieval check (Story 5)

Asks the vector store five questions and shows which chunks come back.

This is the manual verification the story's Definition of Done asks for: five
queries covering five different documents. Read the output and check that each
question's top hits come from the document that actually covers that topic, and
that every chunk shows a real source file, page number and section heading.

## Before you run this

1. You need a `.env` at the repository root containing your OpenAI key:

   ```
   OPEN_AI_API_KEY=sk-...
   ```

   `OPENAI_API_KEY` also works — the code accepts either name.

2. This notebook **reads** the vector store. It never writes to it and never
   re-embeds a document. The only thing sent to OpenAI is the five questions
   themselves — five short strings, a fraction of a cent.

3. The store must already be populated. If `python scripts/check_chroma_collection.py`
   reports 0, restore the shared snapshot first — see `vectorstore/README.md`.

In [ ]:
import sys
from pathlib import Path

# Resolve the repo root so `src` imports work no matter where Jupyter started.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.settings import EMBEDDING_MODEL, RETRIEVAL_TOP_K
from src.retrieval import retriever
from src.vectorstore_client import count_collection

print("repo root:", ROOT)
print("chunks in collection:", count_collection())
print("embedding model:", EMBEDDING_MODEL)
print("chunks returned per query:", RETRIEVAL_TOP_K)

## The five questions

One per document, chosen so the right answer is obvious from the source name.
Change the wording or add more — this cell is the only thing you need to edit.

In [ ]:
# question -> a distinctive piece of the source_file it should be answered from
QUESTIONS = {
    "What edge protection is needed when working on a roof?":
        "working-on-roofs",
    "When does an excavation need shoring or benching?":
        "excavation-safety",
    "Who is allowed to erect scaffolding over five metres high?":
        "scaffolding-in-New-Zealand",
    "What training does an operator of a mobile elevating work platform need?":
        "mobile-elevating-work-platforms",
    "What must be done before removing asbestos from a building?":
        "asbestos-removal",
}

for question in QUESTIONS:
    print("-", question)

## Run them

Each question is embedded with the same model used at ingestion, then matched
against the collection. The chunks are printed nearest first, with the source
they came from.

In [ ]:
results_by_question = {}

for question in QUESTIONS:
    print("=" * 78)
    print(question)
    print("=" * 78)

    results = retriever.retrieve(question)
    results_by_question[question] = results

    print(retriever.format_results(results))
    print()

## Did each question find its document?

A quick pass/fail per question: did any of the retrieved chunks come from the
document the question is about, and which document led the results.

`hit` means the expected document appeared somewhere in the results. A miss is
worth looking at, but it is not automatically a bug — a topic can genuinely be
covered by more than one guide.

In [ ]:
for question, expected in QUESTIONS.items():
    results = results_by_question[question]

    sources = [result["source_file"] for result in results]
    hit = any(expected.lower() in source.lower() for source in sources)

    print(f"{'hit ' if hit else 'MISS'}  {question}")
    print(f"        expected: {expected}")
    print(f"        top hit : {sources[0]} (distance {results[0]['distance']:.4f})")
    print(f"        sources : {len(set(sources))} document(s) across {len(results)} chunks")
    print()

hits = sum(
    any(expected.lower() in result["source_file"].lower() for result in results_by_question[q])
    for q, expected in QUESTIONS.items()
)
print(f"{hits} of {len(QUESTIONS)} questions retrieved their expected document")